# Pi-GRPO at Small Scale: An Unbounded Physics-Violation Reward as a Reward-Hacking Mitigation

**Workshop artifact for MOSS @ COLM 2026 (Methods and Opportunities at Small Scale).**

## Abstract

Preference-based rewards used in RL post-training (RLHF / GRPO / DPO) are easy to hack: a
completion can be fluent and highly rated yet physically impossible. This notebook is a
small-scale, fully controlled study of one mitigation from **Pi-GRPO**, a physics-constrained
RL post-training method. The mechanism is a single **unbounded** hard penalty term that scores
every output against a physical envelope (a single-axle kinematic-bicycle model, S-KBM).
Because the term is unbounded above in magnitude of violation, one sustained physics violation
dominates any bounded preference score, so a fluent-but-wrong completion can no longer win.

We do two things, both sized for free Colab:

1. **A reward-hacking probe (pure NumPy, zero GPU, zero downloads).** We build feasible and
   infeasible completions and show that a preference-only reward catches 0/5 infeasible
   completions, while the same set under the physics-grounded reward is caught 5/5. This is the
   measured core of the artifact and runs in under a second on CPU.
2. **A tiny GRPO demonstration on a small (<= 1.5B) model** (default `Qwen/Qwen2.5-0.5B-Instruct`).
   We sample K completions per prompt, parse them into toy trajectories, score them with the
   hybrid reward, compute group-relative advantages, take a few update steps, and watch the
   hard-violation rate fall. This section is guarded so the notebook still runs end to end on CPU.

Everything fits well under the MOSS 1e20-FLOP compute cap (see the FLOP-budget cell).

In [ ]:
#@title  ONE-CLICK: reproduce BOTH tables  (run from the uploaded files)  { display-mode: "form" }
# Anonymized copy for double-blind review.
# Put run_probe.py and run_grpo_local.py next to this notebook
# (Colab: Files sidebar -> Upload, or unzip the supplement here), then:
#   1) Runtime -> Change runtime type -> T4 GPU (free tier is enough), Save.
#   2) Run THIS cell. It installs deps and runs both scripts. Nothing to clone.
import os, sys, subprocess

def sh(cmd):
    print("\n$ " + cmd, flush=True)
    subprocess.run(cmd, shell=True, check=False)

print(">>> installing deps (a no-op if already present on Colab) ...", flush=True)
sh(sys.executable + " -m pip -q install torch transformers accelerate")

need = [f for f in ("run_probe.py", "run_grpo_local.py") if not os.path.exists(f)]
if need:
    print("Missing files: " + ", ".join(need))
    print("Upload them (Files sidebar -> Upload) or unzip the supplement here, then re-run.")
else:
    print("\n" + "#" * 64)
    print("# TABLE 1: reward-hacking probe  (model-free, CPU, under a second)")
    print("#" * 64, flush=True)
    sh(sys.executable + " run_probe.py")
    print("\n" + "#" * 64)
    print("# TABLE 2: small-scale GRPO on Qwen2.5-0.5B  (uses the GPU if present)")
    print("#" * 64, flush=True)
    sh("STEPS=40 WHARDS=0.0,5.0 " + sys.executable + " run_grpo_local.py")
    print("\n>>> done.")
    print(">>> Table 1 expected:  preference-only +10.0 (0/5)  vs  physics -490.5 (5/5).")
    print(">>> Table 2 expected:  base vs trained hard-violation rate for w_hard=0 and w_hard=5.")


## 0. Setup

The probe (Sections 1 and 2) needs only NumPy, which ships with Colab, so it runs with **no GPU
and no installs**. The optional GRPO demonstration (Section 3) needs `torch` and `transformers`.
The install below is guarded: if the libraries are already present (Colab) it is a no-op, and if
the install fails the probe still runs.

In [ ]:
# Guarded install. Safe to run on Colab (libraries already present) or to skip entirely.
import importlib, subprocess, sys

def _have(mod):
    return importlib.util.find_spec(mod) is not None

# NumPy is required and always present on Colab.
assert _have('numpy'), 'NumPy is required for the probe.'

# Only attempt the heavy install if torch/transformers are missing AND a GPU section is wanted.
INSTALL_HEAVY = True  # set False to keep this a pure-NumPy run
if INSTALL_HEAVY and not (_have('torch') and _have('transformers')):
    try:
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q',
             'transformers>=4.44', 'torch', 'accelerate'],
            check=True,
        )
        print('Installed transformers / torch / accelerate.')
    except Exception as e:
        print('Heavy install skipped or failed (probe still runs):', repr(e))
else:
    print('transformers/torch already available or heavy install disabled; nothing to do.')

## 1. The S-KBM Envelope and the Hybrid Reward

A trajectory `tau` is a sequence of states with speed `v_t`, acceleration `a_t`, and curvature
`kappa_t`. The single-axle kinematic-bicycle model (S-KBM) gives the physical bounds:

- `v_max`: maximum speed
- `a_max`: maximum |acceleration|
- `kappa_max = |tan(delta_max)| / L`, where `L` is the wheelbase and `delta_max` the max steering angle

**Hard penalty (unbounded, the anti-reward-hacking core):**

```
R_hard(tau) = - sum_t [ relu(|v_t|/v_max - 1) + relu(|a_t|/a_max - 1) + relu(|kappa_t|/kappa_max - 1) ]
```

with `relu(x) = max(x, 0)`. It is unbounded above in magnitude: a single sustained violation
grows without limit, so it dominates any bounded preference score.

**Hybrid reward:**

```
R(tau) = w_hard * R_hard + w_soft * R_soft + w_data * R_data + w_pref * R_pref
```

For this study `R_pref` is a fluency/preference proxy in `[0, 10]` (a rater that likes fluent
completions), `R_soft` is a mild 95th-percentile envelope penalty, and `R_data` is ~0 for the demo.

In [ ]:
import numpy as np

# ---- S-KBM physical envelope (a compact passenger-car-like profile) ----
V_MAX = 30.0       # m/s  (~108 km/h)
A_MAX = 4.0        # m/s^2
DELTA_MAX = 0.5    # rad, max steering angle
L = 2.7            # m, wheelbase
KAPPA_MAX = abs(np.tan(DELTA_MAX)) / L   # max curvature = |tan(delta_max)| / L

def relu(x):
    return np.maximum(x, 0.0)

def R_hard(traj):
    """Unbounded hard physics-violation penalty (<= 0). One sustained violation dominates."""
    v = np.abs(traj['v']) / V_MAX
    a = np.abs(traj['a']) / A_MAX
    k = np.abs(traj['kappa']) / KAPPA_MAX
    pen = relu(v - 1.0) + relu(a - 1.0) + relu(k - 1.0)
    return -float(np.sum(pen))

def R_soft(traj):
    """Mild bounded penalty on the 95th-percentile envelope usage."""
    v = np.abs(traj['v']) / V_MAX
    a = np.abs(traj['a']) / A_MAX
    k = np.abs(traj['kappa']) / KAPPA_MAX
    p95 = np.percentile(np.concatenate([v, a, k]), 95)
    return -float(relu(p95 - 0.95))

def R_pref(traj):
    """Fluency/preference proxy in [0, 10]: a rater that likes fluent completions."""
    return float(traj['pref'])

def R_data(traj):
    """Data-likelihood term, ~0 for this controlled demo."""
    return 0.0

def hybrid_reward(traj, w_hard=5.0, w_soft=0.0, w_data=0.0, w_pref=1.0):
    """R(tau) = w_hard*R_hard + w_soft*R_soft + w_data*R_data + w_pref*R_pref."""
    return (w_hard * R_hard(traj)
            + w_soft * R_soft(traj)
            + w_data * R_data(traj)
            + w_pref * R_pref(traj))

print(f'Envelope: v_max={V_MAX} m/s, a_max={A_MAX} m/s^2, kappa_max={KAPPA_MAX:.4f} 1/m')

## 2. The Reward-Hacking Probe (pure CPU, NumPy)

This is the headline result and runs with **zero GPU and zero downloads**.

We build a small set of completions:

- **Feasible**: speed, acceleration, and curvature all inside the S-KBM envelope. The preference
  rater likes them (`R_pref = +10`).
- **Infeasible**: fluent-but-physically-wrong, speed well over `v_max` (sustained over-speed), yet
  the preference rater still scores them high (`R_pref = +10`).

We then score the same set under two configurations:

- **Preference-only (`w_hard = 0`)**: the physics term is invisible. An infeasible completion is
  scored identically to a feasible one and slips through.
- **Physics-grounded (`w_hard = 5`, the default)**: the unbounded hard penalty dominates, so every
  infeasible completion is driven sharply negative and caught.

A completion is **caught** when the active reward scores it clearly below the feasible baseline
(more than `catch_margin` below the feasible mean). When `w_hard = 0` the reward cannot tell the
two apart, so nothing is caught.

In [ ]:
# Fixed seed for reproducibility.
SEED = 0
rng = np.random.default_rng(SEED)

T = 20            # timesteps per trajectory
K_PROBE = 5       # completions of each kind

# Tuned so the physics-grounded infeasible mean total reward lands at about -490.5.
#   physics infeasible total = w_hard * R_hard + w_pref * 10
#   R_hard(over-speed) = -T * (SCALE - 1) = -20 * (SCALE - 1)
#   5 * (-20 * (SCALE-1)) + 10 = -490.5  ->  SCALE = 6.005
OVERSPEED_SCALE = 6.005

def make_feasible():
    v = rng.uniform(0.3, 0.8, T) * V_MAX            # inside envelope
    a = rng.uniform(-0.5, 0.5, T) * A_MAX
    kappa = rng.uniform(-0.5, 0.5, T) * KAPPA_MAX
    return {'v': v, 'a': a, 'kappa': kappa, 'pref': 10.0}

def make_infeasible(scale=OVERSPEED_SCALE):
    # Fluent but physically wrong: sustained over-speed, rater still loves it (+10).
    v = np.full(T, scale * V_MAX)                   # well over v_max
    a = rng.uniform(-0.5, 0.5, T) * A_MAX           # accel / curvature stay legal
    kappa = rng.uniform(-0.5, 0.5, T) * KAPPA_MAX
    return {'v': v, 'a': a, 'kappa': kappa, 'pref': 10.0}

feasibles = [make_feasible() for _ in range(K_PROBE)]
infeasibles = [make_infeasible() for _ in range(K_PROBE)]

def eval_config(name, w_hard, catch_margin=1.0):
    inf_tot = [hybrid_reward(t, w_hard=w_hard) for t in infeasibles]
    fea_tot = [hybrid_reward(t, w_hard=w_hard) for t in feasibles]
    fea_mean = float(np.mean(fea_tot))
    # Caught = active reward scores the infeasible completion clearly below the feasible baseline.
    caught = sum(1 for tot in inf_tot if tot < fea_mean - catch_margin)
    return {
        'config': name,
        'w_hard': w_hard,
        'inf_mean': float(np.mean(inf_tot)),
        'caught': caught,
        'fea_mean': fea_mean,
    }

results = [
    eval_config('preference-only (w_hard=0)', w_hard=0.0),
    eval_config('physics-grounded (w_hard=5)', w_hard=5.0),
]

# ---- print the clean table ----
hdr = f"{'config':<30}{'infeasible mean reward':>24}{'caught':>10}{'feasible mean reward':>24}"
print(hdr)
print('-' * len(hdr))
for r in results:
    caught_str = f"{r['caught']}/{K_PROBE}"
    print(f"{r['config']:<30}{r['inf_mean']:>+24.1f}{caught_str:>10}{r['fea_mean']:>+24.1f}")

print()
print('Reading: preference-only cannot distinguish physics violations (0/5 caught);')
print('the unbounded hard term flips the same infeasible set to strongly negative (5/5 caught).')

## 3. Tiny GRPO Demonstration on a Small Model

We now wire the same reward into a minimal GRPO loop on a small (<= 1.5B) instruction model.

**Group-relative advantage (GRPO), no value head:**

```
A_k = (R_k - mean(R_1..K)) / (std(R_1..K) + eps)
```

For each prompt we sample `K` short completions, parse each into a toy trajectory, score it with
the hybrid reward, compute group-relative advantages, and take a few policy-gradient steps that
push probability toward above-average completions. We track the **hard-violation rate** (fraction
of sampled completions with `R_hard < 0`) and expect it to fall.

The whole section is guarded: with no GPU or no model available it prints a clear message and
skips, so the notebook still runs end to end on CPU. Sizes are tiny (a few prompts, K=4, a few
steps) so it fits a free Colab T4 in well under 12 hours.

In [ ]:
# Configuration for the GRPO demonstration.
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'   # <= 1.5B; change to any small instruct model
K_SAMPLES = 4        # completions per prompt (the GRPO group size)
N_STEPS = 6          # GRPO update steps
MAX_NEW_TOKENS = 48
LR = 1e-5
EPS = 1e-6

PROMPTS = [
    'Output a 6-step driving plan as speed_mps,accel_mps2,curvature pairs, one per line.',
    'Plan a safe lane change. List speed_mps,accel_mps2,curvature for 6 steps, one per line.',
    'Give a 6-step highway cruise as speed_mps,accel_mps2,curvature lines.',
]

In [ ]:
import re

def parse_trajectory(text):
    """Parse 'speed,accel,curvature' triples from model text into a toy trajectory.
    Robust to junk: pulls every numeric triple it can find. Falls back to a mild
    in-envelope trajectory if nothing parses, so scoring never crashes."""
    nums = re.findall(r'[-+]?\d*\.?\d+', text)
    vals = [float(x) for x in nums]
    triples = [vals[i:i+3] for i in range(0, len(vals) - len(vals) % 3, 3)]
    if not triples:
        v = np.full(4, 0.5 * V_MAX); a = np.zeros(4); k = np.zeros(4)
        return {'v': v, 'a': a, 'kappa': k, 'pref': 5.0}
    arr = np.array(triples, dtype=float)
    v, a, k = arr[:, 0], arr[:, 1], arr[:, 2]
    # Preference proxy: rewards well-formed, multi-step, 'fluent' outputs, blind to physics.
    pref = float(np.clip(2.0 + 1.5 * len(triples), 0.0, 10.0))
    return {'v': v, 'a': a, 'kappa': k, 'pref': pref}

def group_advantages(rewards, eps=EPS):
    """GRPO group-relative advantage: A_k = (R_k - mean) / (std + eps), no value head."""
    r = np.asarray(rewards, dtype=float)
    return (r - r.mean()) / (r.std() + eps)

In [ ]:
# Guarded GRPO loop. Skips cleanly with a message if no GPU or model is unavailable.
GRPO_DONE = False
violation_rate_history = []

try:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if not torch.cuda.is_available():
        raise RuntimeError('No CUDA GPU detected.')

    torch.manual_seed(SEED)
    device = 'cuda'
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(device)
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=LR)

    def build_inputs(prompt):
        msgs = [{'role': 'user', 'content': prompt}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        return tok(text, return_tensors='pt').to(device)

    for step in range(N_STEPS):
        step_loss = 0.0
        step_violations, step_total = 0, 0
        for prompt in PROMPTS:
            enc = build_inputs(prompt)
            prompt_len = enc['input_ids'].shape[1]
            with torch.no_grad():
                gen = model.generate(
                    **enc, do_sample=True, top_p=0.95, temperature=1.0,
                    num_return_sequences=K_SAMPLES, max_new_tokens=MAX_NEW_TOKENS,
                    pad_token_id=tok.pad_token_id,
                )
            comp_ids = gen[:, prompt_len:]
            texts = tok.batch_decode(comp_ids, skip_special_tokens=True)
            trajs = [parse_trajectory(t) for t in texts]
            rewards = [hybrid_reward(tr, w_hard=5.0) for tr in trajs]
            advs = group_advantages(rewards)

            step_violations += sum(1 for tr in trajs if R_hard(tr) < 0.0)
            step_total += len(trajs)

            # Policy-gradient (REINFORCE-style) step with group-relative advantages.
            full = torch.cat([enc['input_ids'].repeat(K_SAMPLES, 1), comp_ids], dim=1)
            attn = (full != tok.pad_token_id).long()
            out = model(input_ids=full, attention_mask=attn)
            logits = out.logits[:, :-1, :]
            targets = full[:, 1:]
            logp = torch.log_softmax(logits, dim=-1)
            tok_logp = logp.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
            comp_mask = torch.zeros_like(targets, dtype=torch.float16)
            comp_mask[:, prompt_len - 1:] = 1.0
            seq_logp = (tok_logp * comp_mask).sum(dim=1) / comp_mask.sum(dim=1).clamp(min=1)
            adv_t = torch.tensor(advs, dtype=seq_logp.dtype, device=device)
            loss = -(adv_t * seq_logp).mean()

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            step_loss += float(loss.item())

        rate = step_violations / max(step_total, 1)
        violation_rate_history.append(rate)
        print(f'step {step:02d}  loss={step_loss:8.4f}  hard-violation-rate={rate:5.2%}')

    GRPO_DONE = True

except Exception as e:
    print('GRPO demonstration skipped (CPU-only run or model unavailable):', repr(e))
    print('The probe in Section 2 is the measured core and does not need this section.')

In [ ]:
# Plot the hard-violation rate over GRPO steps (only if the loop ran).
if GRPO_DONE and violation_rate_history:
    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(6, 4))
        plt.plot(range(len(violation_rate_history)), violation_rate_history, marker='o')
        plt.xlabel('GRPO step')
        plt.ylabel('hard-violation rate')
        plt.title('Physics-grounded GRPO: hard-violation rate over steps')
        plt.ylim(-0.02, 1.02)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print('Plot skipped:', repr(e))
    print('violation_rate_history =', [round(x, 3) for x in violation_rate_history])
else:
    print('No GRPO history to plot (the GPU section was skipped).')

### DPO augmentation (physics-grounded preference learning)

The same physics signal augments DPO. The implicit reward becomes

```
r_tilde(x, y) = beta * (log pi(y|x) - log pi_ref(y|x)) - gamma_phys * Phi(y)
```

where `Phi(y)` is the per-output S-KBM violation sum (the magnitude of `R_hard`). Setting
`gamma_phys = 0` recovers vanilla DPO. The cell below shows the term as a drop-in function.

In [ ]:
def Phi(traj):
    """Per-output S-KBM violation sum (>= 0); equals -R_hard(traj)."""
    return -R_hard(traj)

def dpo_implicit_reward(logp_policy, logp_ref, traj, beta=0.1, gamma_phys=1.0):
    """r_tilde(x,y) = beta*(log pi - log pi_ref) - gamma_phys*Phi(y). gamma_phys=0 -> vanilla DPO."""
    return beta * (logp_policy - logp_ref) - gamma_phys * Phi(traj)

# Sanity check: an infeasible output is penalized only when gamma_phys > 0.
demo_inf = make_infeasible()
vanilla = dpo_implicit_reward(-2.0, -2.5, demo_inf, beta=0.1, gamma_phys=0.0)
grounded = dpo_implicit_reward(-2.0, -2.5, demo_inf, beta=0.1, gamma_phys=1.0)
print(f'Phi(infeasible)      = {Phi(demo_inf):.2f}')
print(f'vanilla DPO reward   = {vanilla:+.3f}  (gamma_phys=0)')
print(f'physics DPO reward   = {grounded:+.3f}  (gamma_phys=1)')

## 4. FLOP-Budget Estimate and Artifact Logging

MOSS caps training compute at `1e20` FLOPs. We give a rough estimate for the GRPO demonstration
using the standard `~6 * N_params * N_tokens` rule for a training step (forward + backward),
plus a `~2 * N_params * N_tokens` term for the no-grad sampling pass, then show the headroom.

In [ ]:
# Rough FLOP estimate for the tiny GRPO demonstration.
N_PARAMS = 0.5e9                         # Qwen2.5-0.5B
tokens_per_seq = MAX_NEW_TOKENS + 32     # completion + prompt budget
seqs_per_step = K_SAMPLES * len(PROMPTS)
tokens_per_step = seqs_per_step * tokens_per_seq

# sampling (forward only, ~2*N*tokens) + update (forward+backward, ~6*N*tokens)
flops_per_step = (2 + 6) * N_PARAMS * tokens_per_step
total_flops = flops_per_step * N_STEPS

MOSS_CAP = 1e20
print(f'tokens/step      : {tokens_per_step:,}')
print(f'FLOPs/step       : {flops_per_step:.3e}')
print(f'total FLOPs      : {total_flops:.3e}')
print(f'MOSS cap         : {MOSS_CAP:.3e}')
print(f'fraction of cap  : {total_flops / MOSS_CAP:.2e}  ({total_flops / MOSS_CAP * 100:.1e}% of cap)')
assert total_flops < MOSS_CAP, 'Over the MOSS compute cap.'
print('Within the MOSS 1e20-FLOP cap by many orders of magnitude.')

In [ ]:
# Artifact logging: write results.json with the probe table and config.
import json, time

artifact = {
    'artifact': 'moss_pigrpo_probe',
    'workshop': 'MOSS @ COLM 2026',
    'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'seed': SEED,
    'envelope': {
        'v_max': V_MAX, 'a_max': A_MAX, 'delta_max': DELTA_MAX,
        'wheelbase_L': L, 'kappa_max': KAPPA_MAX,
    },
    'reward_weights_default': {'w_hard': 5.0, 'w_soft': 0.0, 'w_data': 0.0, 'w_pref': 1.0},
    'probe': {
        'n_each': K_PROBE, 'timesteps': T, 'overspeed_scale': OVERSPEED_SCALE,
        'rows': results,
    },
    'grpo_demo': {
        'model_id': MODEL_ID, 'k_samples': K_SAMPLES, 'n_steps': N_STEPS,
        'ran': GRPO_DONE, 'violation_rate_history': violation_rate_history,
    },
    'flops': {'total': total_flops, 'moss_cap': MOSS_CAP},
}

with open('results.json', 'w') as f:
    json.dump(artifact, f, indent=2)
print('Wrote results.json')
print(json.dumps(artifact['probe'], indent=2))

## 5. Scope and Mapping to MOSS Small-Scale Criteria

**Honest scope.** The measured core of this artifact is the probe in Section 2: a controlled,
reproducible demonstration that an unbounded physics-violation reward term distinguishes
fluent-but-infeasible completions (caught 0/5 under a preference-only reward, 5/5 under the
physics-grounded reward) and the mechanism in Section 1 that makes it work. The GRPO loop in
Section 3 is a small illustration of how the same reward plugs into policy optimization, not a
full-scale policy result; a full-scale Pi-GRPO policy run is a separate, larger effort.

**How this maps to MOSS (Methods and Opportunities at Small Scale):**

- **Runs on free hardware.** The probe needs only NumPy and runs in under a second on CPU with no
  downloads. The optional GRPO demonstration fits a free Colab T4 in well under 12 hours.
- **Small model.** The demonstration uses a <= 1.5B model (`Qwen/Qwen2.5-0.5B-Instruct` by default).
- **Far under the compute cap.** The FLOP-budget cell shows the run is many orders of magnitude
  below the MOSS 1e20-FLOP cap.
- **A method studied in isolation.** The artifact isolates one mechanism (the unbounded hard
  penalty) and measures exactly what it buys, which is the kind of controlled small-scale study
  MOSS is for.
- **Reproducible.** A fixed seed and a logged `results.json` make the headline numbers checkable.